In [ ]:
from sysdata.sim.csv_futures_sim_data import csvFuturesSimData

data = csvFuturesSimData()

portfolio_intruments =  ["NASDAQ", "SOFR", "CORN"]
instrument = "NASDAQ"
startDate = "2023-06-23"
endDate = "2023-07-27"


In [64]:
from systems.forecasting import TradingRule, Rules
from systems.provided.rules.ewmac import ewmac_forecast_with_defaults as ewmac

ewmac_5_20 = TradingRule((ewmac, [], dict(Lfast=5, Lslow=20)))
ewmac_20_60 = TradingRule((ewmac, [], dict(Lfast=20, Lslow=60)))
ewmac_32_128 = TradingRule((ewmac, [], dict(Lfast=32, Lslow=128)))
ewmac_60_150 = TradingRule((ewmac, [], dict(Lfast=60, Lslow=150)))
ewmac_100_200 = TradingRule((ewmac, [], dict(Lfast=100, Lslow=200)))

my_rules = Rules(dict(ewmac_5_20=ewmac_5_20, 
                      ewmac_20_60=ewmac_20_60,
                      ewmac_32_128=ewmac_32_128,
                      ewmac_60_150=ewmac_60_150,
                      ewmac_100_200=ewmac_100_200))
print(my_rules.trading_rules()["ewmac_5_20"])

TradingRule; function: <function ewmac_forecast_with_defaults at 0x00000279D5897880>, data: data.daily_prices (args: {}) and other_args: Lfast, Lslow


In [65]:
from systems.basesystem import System
import pandas as pd

my_system = System([my_rules], data)
fc = my_system.rules.get_raw_forecast(instrument, "ewmac_5_20")

filtered = fc[startDate : endDate]
print(filtered)

Private configuration private/private_config.yaml does not exist; no problem if running in sim mode
2025-12-07 10:58:27 DEBUG base_system Following instruments are 'duplicate_markets' ['Another_thing', 'bad_thing'] 
2025-12-07 10:58:27 DEBUG base_system Following instruments are marked as 'ignore_instruments': not included: ['EXAMPLE']
2025-12-07 10:58:27 DEBUG base_system Following instruments removed entirely from sim: ['Another_thing', 'EXAMPLE', 'bad_thing']
2025-12-07 10:58:27 DEBUG base_system {'stage': 'rules', 'instrument_code': 'NASDAQ'} Calculating raw forecast NASDAQ for ewmac_5_20
index
2023-06-23    1.724097
2023-06-26    1.123423
2023-06-27    1.066048
2023-06-28    1.134256
2023-06-29    1.069559
2023-06-30    1.282533
2023-07-03    1.489151
2023-07-04    1.489151
2023-07-05    1.575676
2023-07-06    1.276453
2023-07-07    1.054222
2023-07-10    0.901322
2023-07-11    0.905732
2023-07-12    1.178520
2023-07-13    1.654332
2023-07-14    1.941481
2023-07-17    2.284723
202

In [66]:
rule_names = list(my_rules.trading_rules().keys())

"""
df = pd.DataFrame()

for rule in rule_names:
    df[rule] = my_system.rules.get_raw_forecast(instrument, rule)["2023-06-23" : "2023-07-27"]
"""

'\ndf = pd.DataFrame()\n\nfor rule in rule_names:\n    df[rule] = my_system.rules.get_raw_forecast(instrument, rule)["2023-06-23" : "2023-07-27"]\n'

In [67]:
from sysdata.config.configdata import Config

my_config = Config()
my_config

empty_rules = Rules()
my_config.trading_rules =  Rules(dict(ewmac_5_20=ewmac_5_20, 
                                        ewmac_20_60=ewmac_20_60,
                                        ewmac_32_128=ewmac_32_128,
                                        ewmac_60_150=ewmac_60_150,
                                        ewmac_100_200=ewmac_100_200))

my_system = System([empty_rules], data, my_config)

from systems.forecast_scale_cap import ForecastScaleCap

# we can estimate these ourselves
my_config.instruments = portfolio_intruments
my_config.use_forecast_scale_estimates = True

fcs = ForecastScaleCap()
my_system = System([fcs, my_rules], data, my_config)
my_config.forecast_scalar_estimate["pool_instruments"] = False

Private configuration private/private_config.yaml does not exist; no problem if running in sim mode
Private configuration private/private_config.yaml does not exist; no problem if running in sim mode


In [68]:
from systems.forecast_combine import ForecastCombine

# estimates:
from systems.accounts.accounts_stage import Account
from systems.rawdata import RawData
from systems.positionsizing import PositionSizing

my_account = Account()
combiner = ForecastCombine()
raw_data = RawData()
position_size = PositionSizing()


my_config.forecast_weight_estimate = dict(method="one_period")
my_config.use_forecast_weight_estimates = True
my_config.use_forecast_div_mult_estimates = True

my_system = System(
    [my_account, fcs, my_rules, combiner, raw_data, position_size], data, my_config
)

Private configuration private/private_config.yaml does not exist; no problem if running in sim mode


In [ ]:
df = pd.DataFrame()

for rule in rule_names:
    # raw forecast
    df[f"{rule}_raw"] = my_system.rules.get_raw_forecast(instrument, rule)[startDate : endDate]

for rule in rule_names:
    # scaled + capped forecast
    df[f"{rule}_scaled_capped"] = my_system.forecastScaleCap.get_scaled_forecast(instrument, rule)[startDate : endDate]

for rule in rule_names:
    #weights 
    df[f"{rule}_weight"] = my_system.combForecast.get_forecast_weights(instrument)[rule][startDate : endDate]

for rule in rule_names:
    #combined forecast
    df["combined_forecast"] = my_system.combForecast.get_combined_forecast(instrument)[startDate : endDate]

# ---- Save to CSV ----

import os
os.makedirs("results", exist_ok=True)
outpath = "results/forecasts_" + instrument + ".csv"
df.to_csv(outpath)

print(f"Saved to {outpath}")

2025-12-07 10:58:36 DEBUG base_system Following instruments are 'duplicate_markets' ['Another_thing', 'bad_thing'] 
2025-12-07 10:58:36 DEBUG base_system Following instruments are marked as 'ignore_instruments': not included: ['EXAMPLE']
2025-12-07 10:58:36 DEBUG base_system Following instruments removed entirely from sim: ['Another_thing', 'EXAMPLE', 'bad_thing']
2025-12-07 10:58:36 DEBUG base_system {'stage': 'rules', 'instrument_code': 'NASDAQ'} Calculating raw forecast NASDAQ for ewmac_5_20
2025-12-07 10:58:37 DEBUG base_system {'stage': 'rules', 'instrument_code': 'NASDAQ'} Calculating raw forecast NASDAQ for ewmac_20_60
2025-12-07 10:58:37 DEBUG base_system {'stage': 'rules', 'instrument_code': 'NASDAQ'} Calculating raw forecast NASDAQ for ewmac_32_128
2025-12-07 10:58:37 DEBUG base_system {'stage': 'rules', 'instrument_code': 'NASDAQ'} Calculating raw forecast NASDAQ for ewmac_60_150
2025-12-07 10:58:37 DEBUG base_system {'stage': 'rules', 'instrument_code': 'NASDAQ'} Calculatin

In [60]:
possizer = PositionSizing()
my_config.percentage_vol_target = 25
my_config.notional_trading_capital = 500000
my_config.base_currency = "EUR"

print(my_system.positionSize.get_price_volatility(instrument).tail(5))
print(my_system.positionSize.get_block_value(instrument).tail(5))
print(my_system.positionSize.get_underlying_price(instrument))
print(my_system.positionSize.get_instrument_value_vol(instrument).tail(5))
print(my_system.positionSize.get_average_position_at_subsystem_level(instrument).tail(5))
print(my_system.positionSize.get_vol_target_dict())
print(my_system.positionSize.get_subsystem_position(instrument).tail(5))

# portfolio - estimated
from systems.portfolio import Portfolios

portfolio = Portfolios()

my_config.use_instrument_weight_estimates = True
my_config.use_instrument_div_mult_estimates = True
my_config.instrument_weight_estimate = dict(method="shrinkage", date_method="in_sample")

my_system = System(
    [my_account, fcs, my_rules, combiner, possizer, portfolio, raw_data],
    data,
    my_config,
)

print(my_system.portfolio.get_instrument_weights().tail(5))
print(my_system.portfolio.get_instrument_diversification_multiplier().tail(5))

index
2024-03-22    0.820081
2024-03-25    0.805534
2024-03-26    0.794334
2024-03-27    0.776117
2024-03-28    0.763760
Freq: B, dtype: float64
index
2024-03-22    3712.75
2024-03-25    3706.70
2024-03-26    3696.00
2024-03-27    3701.10
2024-03-28    3693.00
Freq: B, Name: PRICE, dtype: float64
index
1999-12-14     3224.00
1999-12-15     3275.50
1999-12-16     3382.00
1999-12-17     3414.00
1999-12-20     3446.50
                ...   
2024-03-22    18563.75
2024-03-25    18533.50
2024-03-26    18480.00
2024-03-27    18505.50
2024-03-28    18465.00
Freq: B, Name: PRICE, Length: 6338, dtype: float64
2025-12-07 10:46:28 DEBUG base_system {'stage': 'positionSize', 'instrument_code': 'NASDAQ'} Calculating instrument value vol for NASDAQ
2025-12-07 10:46:28 DEBUG base_system {'stage': 'positionSize', 'instrument_code': 'NASDAQ'} Calculating instrument currency vol for NASDAQ
index
2024-03-22    3044.757194
2024-03-25    2985.874293
2024-03-26    2935.859699
2024-03-27    2872.485593
2024-

In [61]:
my_system = System(
    [fcs, my_rules, combiner, possizer, portfolio, my_account, raw_data],
    data,
    my_config,
)
profits = my_system.accounts.portfolio()
print(profits.percent.stats())

Private configuration private/private_config.yaml does not exist; no problem if running in sim mode
2025-12-07 10:47:01 DEBUG base_system Following instruments are 'duplicate_markets' ['Another_thing', 'bad_thing'] 
2025-12-07 10:47:01 DEBUG base_system Following instruments are marked as 'ignore_instruments': not included: ['EXAMPLE']
2025-12-07 10:47:01 DEBUG base_system Following instruments removed entirely from sim: ['Another_thing', 'EXAMPLE', 'bad_thing']
2025-12-07 10:47:01 INFO base_system {'stage': 'accounts'} Calculating pandl for portfolio
2025-12-07 10:47:01 DEBUG base_system {'stage': 'positionSize'} Getting vol target
2025-12-07 10:47:01 DEBUG base_system {'stage': 'accounts', 'instrument_code': 'CORN'} Calculating pandl for instrument for CORN
2025-12-07 10:47:01 DEBUG base_system {'stage': 'portfolio', 'instrument_code': 'CORN'} Calculating notional position for CORN
2025-12-07 10:47:01 INFO base_system {'stage': 'portfolio', 'instrument_code': 'CORN'} Calculating inst

In [62]:
# have costs data now
print(profits.gross.percent.stats())
print(profits.net.percent.stats())

[[('min', '-14.69'), ('max', '11.36'), ('median', '0'), ('mean', '0.02099'), ('std', '1.17'), ('skew', '-0.34'), ('ann_mean', '5.375'), ('ann_std', '18.72'), ('sharpe', '0.2872'), ('sortino', '0.2615'), ('avg_drawdown', '-12.24'), ('time_in_drawdown', '0.4601'), ('calmar', '0.06251'), ('avg_return_to_drawdown', '0.4392'), ('avg_loss', '-1.205'), ('avg_gain', '1.184'), ('gaintolossratio', '0.9826'), ('profitfactor', '1.078'), ('hitrate', '0.5232'), ('t_stat', '2.079'), ('p_value', '0.03761')], ('You can also plot / print:', ['rolling_ann_std', 'drawdown', 'curve', 'percent'])]
[[('min', '-14.7'), ('max', '11.36'), ('median', '0'), ('mean', '0.01795'), ('std', '1.17'), ('skew', '-0.3495'), ('ann_mean', '4.596'), ('ann_std', '18.72'), ('sharpe', '0.2456'), ('sortino', '0.2244'), ('avg_drawdown', '-13.31'), ('time_in_drawdown', '0.4627'), ('calmar', '0.05141'), ('avg_return_to_drawdown', '0.3453'), ('avg_loss', '-1.158'), ('avg_gain', '1.185'), ('gaintolossratio', '1.024'), ('profitfactor'

In [ ]:
profits.net.percent.curve().plot()